# Fix agent endpoint UC perms

Run this notebook in the workspace **after** an agent stage has deployed an endpoint that returns `PermissionError("User does not have EXECUTE on Routine '<catalog>.ai.<fn>'")` on tool calls.

## What it does

1. For each agent endpoint you list, sends one probe chat request to force the runtime to attempt a UC function call.  This generates a 403 `getFunction` audit event whose `user_identity.email` is the endpoint's hidden auto-managed runtime SP UUID — the SP that `agents.deploy()` was supposed to grant `EXECUTE` to but silently didn't.
2. Polls `system.access.audit` until that SP UUID surfaces (typically 1–3 minutes).
3. Grants the SP `USE_CATALOG` + `USE_SCHEMA` + `EXECUTE` on the agent's function schema (and `SELECT` on tables the functions read).  Idempotent.
4. Re-invokes each endpoint to confirm the tool calls now succeed.

## Why this exists

The Mosaic AI Agent Framework's `agents.deploy()` creates an internal runtime SP for `EMBEDDED_CREDENTIALS` endpoints.  That SP is meant to receive automatic `EXECUTE` grants via the `resources=[DatabricksFunction(...)]` list passed to `mlflow.pyfunc.log_model()`, but the auto-grant silently fails in some workspaces (observed 2026-06-08).  The SP is **not visible in workspace SCIM** — the audit log is the only path that surfaces its UUID.  See `utils/agent_runtime_grants.py` for the full backstory.

In [ ]:
dbutils.widgets.text("CATALOG", "", "UC catalog (REQUIRED)")
dbutils.widgets.text(
    "ENDPOINT_NAMES",
    "",
    "Endpoints to fix, comma-separated (default: <catalog>_{refund,complaint,support}_agent)",
)
dbutils.widgets.text("MAX_WAIT_SECONDS", "300", "Max seconds to wait for audit log")

CATALOG = dbutils.widgets.get("CATALOG").strip()
if not CATALOG:
    raise ValueError(
        "CATALOG widget is empty. Set it to the UC catalog name that matches "
        "the `--var catalog=<name>` from your last `bundle deploy` "
        "(bundle default: 'caspersdev')."
    )

_eps = dbutils.widgets.get("ENDPOINT_NAMES").strip()
if _eps:
    ENDPOINTS = [e.strip() for e in _eps.split(",") if e.strip()]
else:
    # Mirrors the endpoint name pattern from databricks.yml job params:
    #   ${var.catalog}_{refund,complaint,support}_agent
    ENDPOINTS = [
        f"{CATALOG}_refund_agent",
        f"{CATALOG}_complaint_agent",
        f"{CATALOG}_support_agent",
    ]

MAX_WAIT = int(dbutils.widgets.get("MAX_WAIT_SECONDS"))

print(f"Catalog:   {CATALOG}")
print(f"Endpoints: {ENDPOINTS}")
print(f"Audit-log wait budget: {MAX_WAIT}s")

In [ ]:
import json
import re
import sys
import time
from typing import List, Optional, Set

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceDoesNotExist

w = WorkspaceClient()

UUID_RE = re.compile(r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$")

# A trivial probe that any of the 3 agents will accept and that always
# forces at least one UC function call (the agents all start with a UC
# lookup before reasoning).  We don't care about the response — we only
# care that the runtime tried a UC call so the audit log captures the SP.
PROBE_PAYLOAD = {
    "messages": [
        {"role": "user", "content": "Look up order ABCDEF and return whatever you find."}
    ]
}

In [ ]:
def probe(endpoint_name: str) -> bool:
    """Send one chat request to the endpoint.  Returns True iff the
    response indicates a UC permission denial (the case we're fixing).
    Returns False if the call already succeeded — nothing to do."""
    try:
        resp = w.serving_endpoints.query(
            name=endpoint_name,
            messages=[{"role": "user", "content": PROBE_PAYLOAD["messages"][0]["content"]}],
        )
        # The SDK returns a typed object; cast through json for uniform inspection.
        body = json.dumps(resp.as_dict())
    except Exception as exc:  # noqa: BLE001
        # Some agent endpoints return 200 with the error in the message body;
        # others raise.  Treat both as "likely the perm bug".
        body = repr(exc)
    if "Permission denied" in body and "EXECUTE on Routine" in body:
        return True
    if "Permission denied" in body or "PermissionError" in body:
        # Different perm error — surface it so user can see, but still try
        # to grant; the audit log will tell us what the runtime tried.
        print(f"  ⚠\ufe0f  endpoint returned a permission-related error; raw fragment:")
        print(f"     {body[:300]}")
        return True
    return False


def discover_runtime_sps(catalog: str, schema: str, since_seconds: int) -> Set[str]:
    """Read system.access.audit for UUID-shaped principals that have
    touched <catalog>.<schema>.* in the last `since_seconds`.  Returns
    the set of distinct SP UUIDs."""
    rows = spark.sql(f"""
        SELECT DISTINCT user_identity.email AS principal
        FROM system.access.audit
        WHERE event_time >= current_timestamp() - INTERVAL {int(since_seconds)} SECONDS
          AND service_name = 'unityCatalog'
          AND request_params.full_name_arg LIKE '{catalog}.{schema}.%'
          AND user_identity.email IS NOT NULL
    """).collect()
    return {r["principal"] for r in rows if r["principal"] and UUID_RE.match(r["principal"])}


def grant_perms(catalog: str, principal: str) -> None:
    """Apply the standard agent runtime grant bundle to one principal.
    Idempotent.  Mirrors what utils/agent_runtime_grants.py does so this
    notebook stays self-contained."""
    stmts = [
        f"GRANT USE CATALOG ON CATALOG {catalog} TO `{principal}`",
        f"GRANT USE SCHEMA ON SCHEMA {catalog}.ai TO `{principal}`",
        f"GRANT EXECUTE  ON SCHEMA {catalog}.ai TO `{principal}`",
        f"GRANT USE SCHEMA ON SCHEMA {catalog}.lakeflow TO `{principal}`",
        f"GRANT SELECT  ON SCHEMA {catalog}.lakeflow TO `{principal}`",
        f"GRANT USE SCHEMA ON SCHEMA {catalog}.simulator TO `{principal}`",
        f"GRANT SELECT  ON SCHEMA {catalog}.simulator TO `{principal}`",
    ]
    ok = err = 0
    for stmt in stmts:
        try:
            spark.sql(stmt)
            ok += 1
        except Exception as exc:  # noqa: BLE001 (schema may not exist in this catalog)
            err += 1
            print(f"     ⚠ {stmt} → {exc}")
    print(f"     {ok} grants ok, {err} skipped")

## Step 1 — probe each endpoint to generate audit events

In [ ]:
needs_fix: List[str] = []
for ep in ENDPOINTS:
    try:
        w.serving_endpoints.get(ep)
    except ResourceDoesNotExist:
        print(f"➖ {ep}: endpoint does not exist in this workspace — skipping")
        continue
    print(f"→ probing {ep} ...")
    if probe(ep):
        print(f"  ❌ perm error confirmed — will fix")
        needs_fix.append(ep)
    else:
        print(f"  ✅ endpoint already works — no action needed")

print()
print(f"Endpoints needing fix: {needs_fix or 'none — you can stop here'}")

## Step 2 — wait for audit log, discover runtime SPs

In [ ]:
discovered: Set[str] = set()
if needs_fix:
    deadline = time.time() + MAX_WAIT
    poll_every = 30  # seconds
    attempt = 0
    while time.time() < deadline:
        attempt += 1
        # Look back ~10 minutes to be generous on audit-log lag drift.
        sps = discover_runtime_sps(CATALOG, "ai", since_seconds=600)
        new = sps - discovered
        discovered |= sps
        if new:
            print(f"  [attempt {attempt}] +{len(new)} new SP(s): {sorted(new)}")
        if len(discovered) >= len(needs_fix):
            # Heuristic: one SP per endpoint; if we have at least that many,
            # we've probably caught them all.  We still poll a bit more in
            # case audit-log ingestion is racing us.
            print(f"  found {len(discovered)} SP(s) — likely complete")
            break
        time.sleep(poll_every)
    else:
        print(f"  ⚠\ufe0f  wait budget ({MAX_WAIT}s) exhausted; proceeding with what we have")

print()
print(f"Discovered runtime SPs: {sorted(discovered) or 'NONE — nothing to grant'}")

## Step 3 — grant the SPs the required UC perms

In [ ]:
for sp in sorted(discovered):
    print(f"→ granting perms to {sp}")
    grant_perms(CATALOG, sp)

## Step 4 — verify by re-invoking each endpoint

In [ ]:
still_broken: List[str] = []
for ep in needs_fix:
    print(f"→ verifying {ep} ...")
    if probe(ep):
        print(f"  ❌ still failing")
        still_broken.append(ep)
    else:
        print(f"  ✅ now works")

print()
if still_broken:
    print(f"⚠\ufe0f  still broken: {still_broken}")
    print("   This usually means either (a) the new audit events have not yet")
    print("   propagated; re-run this notebook in ~5 minutes, or (b) the runtime")
    print("   SP needs different perms than the standard bundle this script grants.")
else:
    print("✅ all probed endpoints now succeed.")